# From Prediction to Causality: Telco Churn Notebook

This notebook creates the worked modelling flow used in the presentation:

1. **Prediction**: train and compare Logistic Regression, Random Forest, SVM and XGBoost.
2. **Model explanation**: use SHAP values to explain the XGBoost model locally and globally.
3. **Attribution**: use Logistic Regression and GAMs to explore associations with churn.
4. **Grouped attribution**: use k-means clustering to create an `inferred_region` variable and compare churn patterns between inferred groups.

The running example is the **IBM / Kaggle Telco Customer Churn** dataset. The key theme throughout is that prediction, explanation, attribution and causality answer different questions.

## 0. Setup

In [ ]:
# Install packages
%pip install pandas numpy matplotlib scikit-learn xgboost shap pygam statsmodels patsy kagglehub

# Hide warnings 
import warnings
warnings.filterwarnings("ignore")

# Core libraries
import os
import glob

import pandas as pd
import numpy as np

# Display
from IPython.display import display, Math

# Visualisation
import matplotlib.pyplot as plt

# Data download
import kagglehub

# Preprocessing and pipelines
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Data splitting and validation
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate

# Unsupervised learning
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from pygam import LogisticGAM, s, f

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    classification_report
)

# Model explanation
import shap

# Statistics and formulas
import statsmodels.api as sm
import statsmodels.formula.api as smf
from patsy import dmatrix

# Sparse matrices
from scipy import sparse

In [ ]:
# Presentation-style plot settings
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 120
})

# Output folder for visuals
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42

## 1. Load the Telco Churn dataset from Kaggle

The base dataset contains customer profile, account, service, charge and churn variables. This is enough for prediction, model explanation and association, but it is not enough on its own to prove that a variable caused churn.

In [ ]:
# Download dataset from Kaggle using kagglehub
# Kaggle dataset: https://www.kaggle.com/datasets/blastchar/telco-customer-churn

dataset_path = kagglehub.dataset_download("blastchar/telco-customer-churn")
print("Dataset downloaded to:", dataset_path)

csv_files = glob.glob(os.path.join(dataset_path, "*.csv"))
print("CSV files found:", csv_files)

if not csv_files:
    raise FileNotFoundError("No CSV file found in the downloaded Kaggle dataset folder.")

df = pd.read_csv(csv_files[0])
display(df.head())
print(df.shape)

### 1.1 Clean the dataset

In [ ]:
# Basic cleaning
df = df.copy()

# Convert TotalCharges to numeric. Some rows contain blank strings.
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Drop rows with missing TotalCharges
df = df.dropna(subset=["TotalCharges"]).reset_index(drop=True)

# Binary target
df["Churn_binary"] = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

# Remove customerID if present
if "customerID" in df.columns:
    df = df.drop(columns=["customerID"])

display(df.head())
print(df.info())
print(df["Churn"].value_counts(normalize=True).rename("proportion"))

## 2. Prediction: compare four classification models

The prediction task asks: **Who is likely to churn?**

We fit four models:

- Logistic Regression
- Support Vector Machine
- Random Forest
- XGBoost

The models are evaluated using **accuracy** and **ROC AUC**.

In [ ]:
# Feature and target setup
target = "Churn_binary"

X = df.drop(columns=["Churn", "Churn_binary"])
y = df[target]

numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [ ]:
# OneHotEncoder compatibility across scikit-learn versions
def make_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", make_ohe(), categorical_features)
    ],
    remainder="drop",
    sparse_threshold=0
)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Support Vector Machine": SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        min_samples_leaf=5,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=400,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
}

fitted_models = {}
results = []

for model_name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", preprocess),
        ("model", model)
    ])

    pipe.fit(X_train, y_train)
    fitted_models[model_name] = pipe

    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1]

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "ROC AUC": roc_auc_score(y_test, y_proba)
    })

results_df = pd.DataFrame(results).sort_values("ROC AUC", ascending=False).reset_index(drop=True)
display(results_df)

In [ ]:
# Save model comparison table
results_df.to_csv(os.path.join(OUTPUT_DIR, "model_comparison_accuracy_auc.csv"), index=False)

# A PowerPoint-friendly rounded table
model_table = results_df.copy()
model_table["Accuracy"] = model_table["Accuracy"].round(3)
model_table["ROC AUC"] = model_table["ROC AUC"].round(3)
display(model_table)

## 3. Model explanation: SHAP values for XGBoost

The model explanation task asks: **Why did the model make this prediction?**

SHAP values explain how each feature contributed to a model prediction. They can be used locally for one customer or globally across many customers.

Important caution: **SHAP explains the model's reasoning, not the real-world cause of churn.**

In [ ]:
# Extract the fitted XGBoost pipeline and processed feature matrix
xgb_pipe = fitted_models["XGBoost"]
xgb_model = xgb_pipe.named_steps["model"]
xgb_preprocess = xgb_pipe.named_steps["preprocess"]

X_train_processed = xgb_preprocess.transform(X_train)
X_test_processed = xgb_preprocess.transform(X_test)

raw_feature_names = xgb_preprocess.get_feature_names_out()

friendly_names = {
    "gender": "Gender",
    "SeniorCitizen": "Senior citizen",
    "Partner": "Partner",
    "Dependents": "Dependents",
    "tenure": "Tenure",
    "PhoneService": "Phone service",
    "MultipleLines": "Multiple lines",
    "InternetService": "Internet service",
    "OnlineSecurity": "Online security",
    "OnlineBackup": "Online backup",
    "DeviceProtection": "Device protection",
    "TechSupport": "Tech support",
    "StreamingTV": "Streaming TV",
    "StreamingMovies": "Streaming movies",
    "Contract": "Contract",
    "PaperlessBilling": "Paperless billing",
    "PaymentMethod": "Payment method",
    "MonthlyCharges": "Monthly charges",
    "TotalCharges": "Total charges"
}

def clean_feature_name(name):
    name = name.replace("num__", "").replace("cat__", "")
    for col in categorical_features:
        prefix = f"{col}_"
        if name.startswith(prefix):
            feature = friendly_names.get(col, col)
            level = name[len(prefix):]
            return f"{feature} = {level}"
    return friendly_names.get(name, name)

feature_names = [clean_feature_name(name) for name in raw_feature_names]

X_train_shap = pd.DataFrame(X_train_processed, columns=feature_names, index=X_train.index)
X_test_shap = pd.DataFrame(X_test_processed, columns=feature_names, index=X_test.index)

display(X_test_shap.head())

In [ ]:
# Compute SHAP values
# We use a background sample to make the plots quicker and stable.
background = shap.sample(X_train_shap, 300, random_state=RANDOM_STATE)

try:
    explainer = shap.TreeExplainer(
        xgb_model,
        data=background,
        feature_perturbation="interventional",
        model_output="probability"
    )
    shap_values = explainer(X_test_shap)
except Exception as e:
    print("Probability-scale TreeExplainer failed; falling back to default SHAP explainer.")
    print("Reason:", e)
    explainer = shap.Explainer(xgb_model, background)
    shap_values = explainer(X_test_shap)

# If SHAP returns a 3D object for binary classification, keep class 1.
if len(shap_values.values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print(shap_values)

In [ ]:
def standardise_shap_plot_fonts(ax, feature_size=10, axis_size=10, title_size=13):
    for label in ax.get_yticklabels():
        label.set_fontsize(feature_size)
        label.set_fontweight("normal")
        label.set_fontstyle("normal")

    for label in ax.get_xticklabels():
        label.set_fontsize(axis_size)
        label.set_fontweight("normal")
        label.set_fontstyle("normal")

    for text in ax.texts:
        text.set_fontweight("normal")
        text.set_fontstyle("normal")

    ax.title.set_fontsize(title_size)
    ax.title.set_fontweight("normal")

### 3.1 Waterfall plot: local explanation for one customer

In [ ]:
# Choose one high-risk customer for the waterfall plot
xgb_test_proba = xgb_pipe.predict_proba(X_test)[:, 1]
customer_position = int(np.argmax(xgb_test_proba))
customer_index = X_test.index[customer_position]

print("Selected customer index:", customer_index)
print("Predicted churn probability:", round(xgb_test_proba[customer_position], 3))
display(X_test.loc[[customer_index]])

plt.figure(figsize=(8, 6))
shap.plots.waterfall(shap_values[customer_position], max_display=15, show=False)
ax = plt.gca()
ax.set_title("Waterfall Plot: One Customer", fontsize=13, fontweight="normal")
standardise_shap_plot_fonts(ax)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_waterfall_one_customer.png"), dpi=300, bbox_inches="tight")
plt.show()

### 3.2 Beeswarm plot: global explanation across customers

In [ ]:
plt.figure(figsize=(8, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
ax = plt.gca()
ax.set_title("Beeswarm Plot: Multiple Customers", fontsize=13, fontweight="normal")
standardise_shap_plot_fonts(ax)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "shap_beeswarm_multiple_customers.png"), dpi=300, bbox_inches="tight")
plt.show()

## 4. Attribution: Logistic Regression and odds ratios

The attribution task asks: **Which factors are associated with churn, after accounting for other variables?**

Logistic regression is useful here because churn is a yes/no outcome. Each coefficient can be converted into an **odds ratio**:

- Odds ratio above 1: associated with higher churn odds.
- Odds ratio below 1: associated with lower churn odds.
- Odds ratio equal to 1: no association relative to the reference.

This remains association, not causation.

In [ ]:
# Create a modelling dataframe for statsmodels
model_df = df.copy()

# Formula kept interpretable for presentation
logit_formula = """
Churn_binary ~ tenure + MonthlyCharges + TotalCharges + SeniorCitizen
+ C(Partner) + C(Dependents)
+ C(InternetService) + C(OnlineSecurity) + C(TechSupport)
+ C(Contract) + C(PaperlessBilling) + C(PaymentMethod)
"""

logit_model = smf.logit(logit_formula, data=model_df).fit(disp=False, maxiter=500)
print(logit_model.summary())

In [ ]:
def tidy_odds_ratios(model, term_labels=None):
    params = model.params
    conf = model.conf_int()
    pvalues = model.pvalues

    out = pd.DataFrame({
        "term": params.index,
        "estimate": params.values,
        "OR": np.exp(params.values),
        "CI_low": np.exp(conf[0].values),
        "CI_high": np.exp(conf[1].values),
        "p_value": pvalues.values
    })

    out = out[out["term"] != "Intercept"].copy()

    if term_labels is not None:
        out["label"] = out["term"].replace(term_labels)
    else:
        out["label"] = out["term"]

    # Make statsmodels terms more readable
    out["label"] = (
        out["label"]
        .str.replace("C\\(", "", regex=True)
        .str.replace("\\)", "", regex=True)
        .str.replace("\\[T\\.", " = ", regex=True)
        .str.replace("\\]", "", regex=True)
        .str.replace("MonthlyCharges", "Monthly charges", regex=False)
        .str.replace("TotalCharges", "Total charges", regex=False)
        .str.replace("tenure", "Tenure", regex=False)
        .str.replace("SeniorCitizen", "Senior citizen", regex=False)
        .str.replace("InternetService", "Internet service", regex=False)
        .str.replace("OnlineSecurity", "Online security", regex=False)
        .str.replace("TechSupport", "Tech support", regex=False)
        .str.replace("PaperlessBilling", "Paperless billing", regex=False)
        .str.replace("PaymentMethod", "Payment method", regex=False)
    )

    out["abs_log_or"] = np.abs(np.log(out["OR"]))
    return out.sort_values("abs_log_or", ascending=False)

or_df = tidy_odds_ratios(logit_model)
display(or_df.head(20))

In [ ]:
def plot_odds_ratios(or_data, title, top_n=15, save_name=None):
    plot_df = or_data.head(top_n).copy()
    plot_df = plot_df.sort_values("OR")

    fig, ax = plt.subplots(figsize=(8, 6))
    y_pos = np.arange(len(plot_df))

    ax.errorbar(
        plot_df["OR"],
        y_pos,
        xerr=[
            plot_df["OR"] - plot_df["CI_low"],
            plot_df["CI_high"] - plot_df["OR"]
        ],
        fmt="o",
        capsize=3
    )

    ax.axvline(1, linestyle="--", linewidth=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(plot_df["label"])
    ax.set_xscale("log")
    ax.set_xlabel("Odds ratio, log scale")
    ax.set_title(title)
    ax.grid(axis="x", linestyle=":", alpha=0.4)

    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=300, bbox_inches="tight")
    plt.show()

plot_odds_ratios(
    or_df,
    title="Logistic Regression Odds Ratios: Association with Churn",
    top_n=15,
    save_name="logistic_regression_odds_ratios.png"
)

## 5. From Logistic Regression to GAMs

Logistic regression can estimate controlled associations, but it often imposes a simple linear shape on numeric predictors.

A **Generalised Additive Model** keeps the binary outcome but allows flexible smooth curves. Here, we compare the predicted churn probability across tenure for:

- Logistic Regression
- GAM

In [ ]:
# Prepare a baseline row for prediction curves
def mode_value(series):
    return series.mode(dropna=True).iloc[0]

baseline = {
    "tenure": model_df["tenure"].median(),
    "MonthlyCharges": model_df["MonthlyCharges"].median(),
    "TotalCharges": model_df["TotalCharges"].median(),
    "SeniorCitizen": int(model_df["SeniorCitizen"].mode().iloc[0]),
    "Partner": mode_value(model_df["Partner"]),
    "Dependents": mode_value(model_df["Dependents"]),
    "InternetService": mode_value(model_df["InternetService"]),
    "OnlineSecurity": mode_value(model_df["OnlineSecurity"]),
    "TechSupport": mode_value(model_df["TechSupport"]),
    "Contract": mode_value(model_df["Contract"]),
    "PaperlessBilling": mode_value(model_df["PaperlessBilling"]),
    "PaymentMethod": mode_value(model_df["PaymentMethod"])
}

tenure_grid = np.linspace(model_df["tenure"].min(), model_df["tenure"].max(), 100)

logit_grid = pd.DataFrame([baseline] * len(tenure_grid))
logit_grid["tenure"] = tenure_grid
logit_pred = logit_model.predict(logit_grid)

plt.figure(figsize=(8, 5))
plt.plot(tenure_grid, logit_pred, linewidth=2)
plt.title("Logistic Regression: Predicted Churn Probability by Tenure")
plt.xlabel("Tenure")
plt.ylabel("Predicted churn probability")
plt.ylim(0, 1)
plt.grid(True, linestyle=":", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "logistic_predicted_probability_tenure.png"), dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Prepare data for GAM
gam_features = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "SeniorCitizen",
    "Partner",
    "Dependents",
    "InternetService",
    "OnlineSecurity",
    "TechSupport",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

gam_df = model_df[gam_features + ["Churn_binary"]].copy()

categorical_gam_features = [
    "Partner",
    "Dependents",
    "InternetService",
    "OnlineSecurity",
    "TechSupport",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

gam_category_maps = {}
for col in categorical_gam_features:
    gam_df[col] = pd.Categorical(gam_df[col])
    gam_category_maps[col] = list(gam_df[col].cat.categories)
    gam_df[col] = gam_df[col].cat.codes

X_gam = gam_df[gam_features].values
y_gam = gam_df["Churn_binary"].values

# Terms:
# continuous terms use s()
# categorical/factor terms use f()
gam_terms = (
    s(0) +               # tenure
    s(1) +               # MonthlyCharges
    s(2) +               # TotalCharges
    f(3) +               # SeniorCitizen
    f(4) + f(5) +        # Partner, Dependents
    f(6) + f(7) + f(8) + # InternetService, OnlineSecurity, TechSupport
    f(9) + f(10) + f(11) # Contract, PaperlessBilling, PaymentMethod
)

gam = LogisticGAM(gam_terms).fit(X_gam, y_gam)
gam.summary()

In [ ]:
# Create equivalent prediction grid for GAM
gam_baseline = gam_df[gam_features].median(numeric_only=True).to_dict()

# For categorical coded variables, use modal category code
for col in categorical_gam_features:
    gam_baseline[col] = int(gam_df[col].mode().iloc[0])

gam_curve_df = pd.DataFrame([gam_baseline] * len(tenure_grid))
gam_curve_df["tenure"] = tenure_grid

gam_pred = gam.predict_mu(gam_curve_df[gam_features].values)

plt.figure(figsize=(8, 5))
plt.plot(tenure_grid, logit_pred, linewidth=2, label="Logistic regression")
plt.plot(tenure_grid, gam_pred, linewidth=2, linestyle="--", label="GAM")
plt.title("Predicted Churn Probability by Tenure: Logistic Regression vs GAM")
plt.xlabel("Tenure")
plt.ylabel("Predicted churn probability")
plt.ylim(0, 1)
plt.legend(frameon=False)
plt.grid(True, linestyle=":", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "logistic_vs_gam_tenure_curve.png"), dpi=300, bbox_inches="tight")
plt.show()

## 6. Grouped attribution: create an inferred region with k-means

The Telco dataset does not contain a natural grouping variable such as region, branch or account manager.

To illustrate the idea behind grouped attribution, we create a simple artificial grouping using k-means clustering. The resulting label is called `inferred_region`.

This is a teaching device: in a real project, the grouping variable should represent a meaningful shared context.

In [ ]:
# Use the same feature matrix as the prediction models for clustering
cluster_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", make_ohe(), categorical_features)
    ],
    remainder="drop",
    sparse_threshold=0
)

X_cluster = cluster_preprocess.fit_transform(X)

kmeans = KMeans(n_clusters=2, random_state=RANDOM_STATE, n_init=20)
clusters = kmeans.fit_predict(X_cluster)

# Label the higher-churn cluster as inferred_region_A for presentation clarity
cluster_summary = pd.DataFrame({
    "cluster": clusters,
    "Churn_binary": y.values
}).groupby("cluster")["Churn_binary"].mean()

high_churn_cluster = cluster_summary.idxmax()

df["cluster"] = clusters
df["inferred_region"] = np.where(
    df["cluster"] == high_churn_cluster,
    "inferred_region_A",
    "inferred_region_B"
)

display(cluster_summary.rename("churn_rate"))
display(df["inferred_region"].value_counts())
display(df.groupby("inferred_region")["Churn_binary"].agg(["mean", "count"]).rename(columns={"mean": "churn_rate"}))

### 6.1 PCA plot to visualise cluster separation

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster)

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "inferred_region": df["inferred_region"]
})

plt.figure(figsize=(8, 6))
for region in sorted(pca_df["inferred_region"].unique()):
    subset = pca_df[pca_df["inferred_region"] == region]
    plt.scatter(
        subset["PC1"],
        subset["PC2"],
        s=18,
        alpha=0.6,
        label=region
    )

plt.title("K-means Clusters Visualised with PCA")
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(frameon=False)
plt.grid(True, linestyle=":", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_inferred_regions.png"), dpi=300, bbox_inches="tight")
plt.show()

## 7. Compare inferred regions with odds ratios

Here we include `inferred_region` in a logistic regression model to estimate whether one inferred group has higher or lower odds of churn after accounting for customer-level variables.

In [ ]:
region_formula = """
Churn_binary ~ C(inferred_region) + tenure + MonthlyCharges + TotalCharges + SeniorCitizen
+ C(Partner) + C(Dependents)
+ C(InternetService) + C(OnlineSecurity) + C(TechSupport)
+ C(Contract) + C(PaperlessBilling) + C(PaymentMethod)
"""

region_model = smf.logit(region_formula, data=df).fit(disp=False, maxiter=500)
region_or_df = tidy_odds_ratios(region_model)

display(region_or_df.head(20))

plot_odds_ratios(
    region_or_df,
    title="Odds Ratios Including Inferred Region",
    top_n=15,
    save_name="odds_ratios_inferred_region.png"
)

In [ ]:
# Display only the inferred region term(s)
region_terms = region_or_df[region_or_df["term"].str.contains("inferred_region", regex=False)]
display(region_terms[["label", "OR", "CI_low", "CI_high", "p_value"]])

## 8. Predicted probability curves by inferred region

Finally, we compare the relationship between **monthly charges** and predicted churn probability across the two inferred regions.

This visual mirrors the grouped-attribution idea: the same customer-level predictor can be shown separately by group.

In [ ]:
monthly_region_formula = """
Churn_binary ~ MonthlyCharges * C(inferred_region)
+ tenure + TotalCharges + SeniorCitizen
+ C(Partner) + C(Dependents)
+ C(InternetService) + C(OnlineSecurity) + C(TechSupport)
+ C(Contract) + C(PaperlessBilling) + C(PaymentMethod)
"""

monthly_region_model = smf.logit(monthly_region_formula, data=df).fit(disp=False, maxiter=500)

monthly_grid = np.linspace(df["MonthlyCharges"].quantile(0.02), df["MonthlyCharges"].quantile(0.98), 100)

curve_rows = []
for region in sorted(df["inferred_region"].unique()):
    temp = pd.DataFrame([baseline] * len(monthly_grid))
    temp["MonthlyCharges"] = monthly_grid
    temp["inferred_region"] = region
    # Maintain a sensible TotalCharges relationship at baseline tenure
    temp["TotalCharges"] = temp["MonthlyCharges"] * temp["tenure"]
    temp["predicted_probability"] = monthly_region_model.predict(temp)
    curve_rows.append(temp[["MonthlyCharges", "inferred_region", "predicted_probability"]])

monthly_curve_df = pd.concat(curve_rows, ignore_index=True)

plt.figure(figsize=(8, 5))
for region in sorted(monthly_curve_df["inferred_region"].unique()):
    subset = monthly_curve_df[monthly_curve_df["inferred_region"] == region]
    plt.plot(
        subset["MonthlyCharges"],
        subset["predicted_probability"],
        linewidth=2,
        label=region
    )

plt.title("Predicted Churn Probability by Monthly Charges and Inferred Region")
plt.xlabel("Monthly charges")
plt.ylabel("Predicted churn probability")
plt.ylim(0, 1)
plt.legend(frameon=False)
plt.grid(True, linestyle=":", alpha=0.4)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "monthly_charges_curves_by_inferred_region.png"), dpi=300, bbox_inches="tight")
plt.show()

## 9. Final notebook summary

This notebook recreated the modelling flow used in the presentation:

- **Prediction**: four supervised classifiers were compared using accuracy and ROC AUC.
- **Model explanation**: SHAP values explained the XGBoost model locally and globally.
- **Attribution**: logistic regression odds ratios were used for controlled association.
- **Nonlinear attribution**: a GAM was used to compare flexible churn curves against logistic regression.
- **Grouped attribution**: k-means created an `inferred_region` label, which was then used to explore group-level differences in churn.

The most important interpretation rule remains:

> Prediction explains risk; SHAP explains the model; attribution describes controlled association; causal inference requires additional design assumptions.